> **이 파일은 실습 버전입니다.**
> `______` 표시가 빈칸입니다. 해설의 핵심 코드를 직접 작성해 보세요.
>
> - `______` → 실행하면 `NameError`가 발생합니다. 정답 코드로 교체하세요.
> - `"______"` → 빈 문자열입니다. 적절한 값을 입력하세요.
> - `# TODO:` 뒤의 힌트를 참고하세요.


# 1. RAG 개요 및 검색 이론

## 학습 목표
- LLM의 환각(Hallucination) 문제를 이해한다
- RAG(Retrieval-Augmented Generation)의 필요성을 학습한다
- 키워드 검색 vs 벡터(시맨틱) 검색의 차이를 이해한다
- 임베딩(Embedding)의 원리와 활용을 학습한다

## 1.1 LLM의 한계와 RAG의 필요성
### LLM(Large Language Model)이란?
- 대규모 텍스트 데이터로 학습된 언어 모델
- GPT-4, Claude, Gemini 등
- 자연어 이해, 생성, 번역, 요약 등 다양한 태스크 수행

### LLM의 한계

| 한계 | 설명 |
|------|------|
| **환각 (Hallucination)** | 사실이 아닌 정보를 자신있게 생성 |
| **지식 단절 (Knowledge Cutoff)** | 학습 데이터 이후 정보 모름 |
| **출처 불명** | 답변의 근거를 제시하기 어려움 |
| **도메인 지식 부족** | 특정 기업/조직의 내부 정보 모름 |

### 환각(Hallucination) 예시

```
질문: "삼성SDS의 2024년 4분기 매출은 얼마인가요?"

LLM 답변 (환각): "삼성SDS의 2024년 4분기 매출은 약 78조원입니다."
→ 실제 데이터가 없어도 그럴듯한 숫자를 생성
```

```
질문: "우리 회사 휴가 정책이 어떻게 되나요?"

LLM 답변 (환각): "일반적으로 연차 15일이 부여됩니다."
→ 회사 내부 규정을 모르면서 일반론으로 답변
```

| **비용 효율** | 모델 재학습 불필요 |

### RAG (Retrieval-Augmented Generation)

**정의**: 검색(Retrieval)과 생성(Generation)을 결합한 기술

### RAG의 장점

| 장점 | 설명 |
|------|------|
| **환각 감소** | 실제 문서 기반 답변 생성 |
| **최신 정보 반영** | 문서 업데이트로 즉시 반영 |
| **출처 제공** | 참조한 문서 명시 가능 |
| **도메인 특화** | 내부 문서로 전문 지식 활용 |
| **비용 효율** | 모델 재학습 불필요 |

## 1.2 키워드 검색 vs 벡터 검색
### 키워드 검색 (Lexical Search)

**원리**: 문서에 포함된 단어(키워드)를 기반으로 검색

**역색인 (Inverted Index)**
```
문서 1: "Python은 프로그래밍 언어입니다"
문서 2: "Java도 프로그래밍 언어입니다"
문서 3: "Python으로 데이터 분석을 합니다"
```

**역색인 구조:**

| 단어 | 문서 ID |
|------|---------|
| Python | [1, 3] |
| Java | [2] |
| 프로그래밍 | [1, 2] |
| 언어 | [1, 2] |
| 데이터 | [3] |
| 분석 | [3] |

**BM25 알고리즘**
- TF (Term Frequency): 문서 내 단어 출현 빈도
- IDF (Inverse Document Frequency): 전체 문서에서의 희소성
- 문서 길이 정규화

**장점**
- 정확한 키워드 매칭에 강함
- 계산 속도 빠름
- 결과 해석 용이

**단점**
- 동의어 처리 불가 ("자동차" ≠ "차량")
- 의미적 유사성 파악 어려움
- 오타에 취약

### 벡터 검색 (Semantic Search)

**원리**: 텍스트를 고차원 벡터로 변환하여 유사도로 검색

**유사도 측정**
- 코사인 유사도 (Cosine Similarity)
- 유클리드 거리 (Euclidean Distance)
- 내적 (Dot Product)

**장점**
- 동의어, 유사어 처리 가능
- 의미적 유사성 파악
- 다국어 지원 가능

**단점**
- 임베딩 생성 비용/시간
- 정확한 키워드 매칭에 약함
- 블랙박스 (해석 어려움)

### 비교 요약

| 구분 | 키워드 검색 | 벡터 검색 |
|------|------------|----------|
| 원리 | 단어 매칭 | 의미 유사도 |
| 알고리즘 | BM25, TF-IDF | kNN, ANN |
| 동의어 | ✗ | ✓ |
| 정확도 | 키워드에 강함 | 의미에 강함 |
| 속도 | 빠름 | 상대적 느림 |
| 비용 | 낮음 | 임베딩 비용 |

### 하이브리드 검색
- 키워드 + 벡터 검색 결합
- 두 방식의 장점 활용
- RRF (Reciprocal Rank Fusion) 등 결합 알고리즘 사용

## 1.3 임베딩(Embedding)의 원리
### 임베딩이란?
텍스트, 이미지 등을 **고차원 벡터**(숫자 배열)로 변환하는 기술

### 왜 벡터로 변환하는가?
- 컴퓨터는 텍스트를 직접 이해할 수 없음
- 숫자로 변환해야 수학적 연산(유사도 계산) 가능
- 의미가 유사한 텍스트는 유사한 벡터로 변환됨

### 임베딩 모델 종류

| 모델 | 제공사 | 차원 | 특징 |
|------|--------|------|------|
| text-embedding-3-small | OpenAI | 1536 | 빠르고 저렴 |
| text-embedding-3-large | OpenAI | 3072 | 고성능 |
| embed-multilingual-v3 | Cohere | 1024 | 다국어 |
| all-MiniLM-L6-v2 | Sentence-Transformers | 384 | 오픈소스, 경량 |
| multilingual-e5-large | Microsoft | 1024 | 오픈소스, 다국어 |

## 1.4 실습: 임베딩 생성 및 유사도 계산
### Colab 환경 설정
먼저 필요한 패키지를 설치하고 환경변수를 설정합니다.

In [ ]:
# [01-01] 필요 패키지 설치
# Colab에서 필요한 패키지 설치
!pip install -q openai python-dotenv numpy

In [ ]:
# [01-02] 라이브러리 임포트
import os
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

In [ ]:
# [01-03] 환경 변수 로드
# .env 파일이 있으면 로드, 없으면 Colab secrets 또는 직접 입력
load_dotenv()

# Colab에서는 아래 주석을 해제하고 API 키 입력
# os.environ["OPENAI_API_KEY"] = "sk-your-api-key"

In [ ]:
# [01-04] OpenAI 클라이언트 생성
client = OpenAI()
print("OpenAI 클라이언트 초기화 완료")

In [ ]:
# [01-05] get_embedding 함수 정의
def get_embedding(text: str, model: str = "text-embedding-3-small") -> list[float]:
    """텍스트를 임베딩 벡터로 변환"""
    response = ______  # TODO: client.embeddings.create() 호출 (input, model, dimensions 파라미터)
    return response.data[0].embedding

# 테스트
embedding = get_embedding("안녕하세요")
print(f"벡터 차원: {len(embedding)}")
print(f"벡터 샘플: {embedding[:5]}")

In [ ]:
# [01-06] cosine_similarity 함수 정의
def cosine_similarity(vec1: list[float], vec2: list[float]) -> float:
    """코사인 유사도 계산"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return ______  # TODO: 코사인 유사도 공식 = dot(a,b) / (norm(a) * norm(b))

In [ ]:
# [01-07] 의미적 유사도 테스트
# 의미적 유사도 테스트
sentences = [
    "Python은 프로그래밍 언어입니다",
    "파이썬으로 코딩을 합니다",        # 유사한 의미
    "오늘 날씨가 좋습니다",            # 다른 의미
    "Java는 프로그래밍 언어입니다",    # 유사한 의미
]

# 임베딩 생성
embeddings = ______  # TODO: 리스트 컴프리헨션으로 모든 문장의 임베딩 생성

# 첫 번째 문장과의 유사도 계산
print(f"기준: \"{sentences[0]}\"\n")
for i in range(1, len(sentences)):
    sim = cosine_similarity(embeddings[0], embeddings[i])
    print(f"유사도 {sim:.4f}: \"{sentences[i]}\"")

### 결과 해석
- "파이썬으로 코딩을 합니다" → 높은 유사도 (동의어 인식)
- "Java는 프로그래밍 언어입니다" → 높은 유사도 (같은 도메인)
- "오늘 날씨가 좋습니다" → 낮은 유사도 (다른 주제)

In [ ]:
# [01-08] 키워드 검색의 한계 시연
# 키워드 검색의 한계 시연
query = "______"  # TODO: 검색어를 입력하세요 (예: "코딩 언어")
documents = [
    "Python은 프로그래밍 언어입니다",
    "Java로 개발을 합니다",
    "JavaScript는 웹 개발에 사용됩니다",
]

print(f"검색어: \"{query}\"\n")

# 키워드 매칭 (단순 포함 여부)
print("[키워드 검색]")
for doc in documents:
    match = ______  # TODO: "코딩" 또는 "언어"가 doc에 포함되어 있는지 확인 (in 연산자)
    print(f"  {'✓' if match else '✗'} {doc}")

print()

# 벡터 검색
print("[벡터 검색]")
query_emb = ______  # TODO: 검색어(query)를 임베딩 벡터로 변환
for doc in documents:
    doc_emb = get_embedding(doc)
    sim = ______  # TODO: query_emb와 doc_emb의 코사인 유사도 계산
    print(f"  {sim:.4f} {doc}")

## 1.5 정리
### 핵심 개념

| 개념 | 설명 |
|------|------|
| LLM 환각 | 사실이 아닌 정보를 생성하는 문제 |
| RAG | 검색 + 생성을 결합하여 환각 감소 |
| 키워드 검색 | 단어 매칭 기반, BM25 알고리즘 |
| 벡터 검색 | 의미 유사도 기반, 임베딩 활용 |
| 임베딩 | 텍스트를 고차원 벡터로 변환 |

### 다음 학습
- OpenSearch 환경 설정
- 텍스트 데이터 인덱싱